In [ ]:
# MIT License
#
#@title Copyright (c) 2026 CCAI Community Authors { display-mode: "form" }
#
# Permission is hereby granted, free of charge, to any person obtaining a
# copy of this software and associated documentation files (the "Software"),
# to deal in the Software without restriction, including without limitation
# the rights to use, copy, modify, merge, publish, distribute, sublicense,
# and/or sell copies of the Software, and to permit persons to whom the
# Software is furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in
# all copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL
# THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING
# FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER
# DEALINGS IN THE SOFTWARE.

# CCAI Tutorial on flood monitoring Part I: People
Authors:
*   Casper Fibaek, [ESA Φ-lab](https://philab.esa.int/), Casper.Fibaek@esa.int
*   Andreas Luyts, [ESA Φ-lab](https://philab.esa.int/), Andreas.Luyts@ext.esa.int
*   Nirdesh Kumar Sharma [Earthsense Labs](https://earthsenselabs.com/), nirdesh@earthsenselabs.com


**Prerequisites**:
1.   Basic understanding of Deep Learning and Remote Sensing.
2.   Familiarity with Python and geospatial data processing.

**Software requirements**:

We are going to use Google Colab. Some cloud storage is necessary to load the imagery.

**Goals**

* The purpose of this tutorial is to give an introduction to the use of Convolutional Neural Networks and Satellite Images for the climate science.
* The data will not be polished RGB data, which is often what is used in AI tutorials, but satellite data that comes with higher dimensions and modalities. The tutorial will introduce these datasets and showcase how they can be used to map structures and surface water in a simple way.

# Table of Contents


*   [Overview](#overview)
*   [Learning Objectives](#learning-objectives)
*   [Climate Impact](#climate-impact)
*   [Target Audience](#target-audience)
*   [Background & Prerequisites](#background-and-prereqs)
*   [Software Requirements](#software-requirements)
*   [Download the Data](#download-data)
*   [Data Description](#data-description)
*   [Methodology](#methodology)
*   [Results & Discussion](#results-and-discussion)
*   [The Carbon Cost of This Notebook](#carbon-cost)
*   [From Predictions to Decisions](#predictions-to-decisions)
*   [Limitations & Responsible Use](#limitations-responsible-use)
*   [References](#references)


<a name="overview"></a>
# Overview

Welcome to this tutorial on flood monitoring and management using Earth Observation and Artificial Intelligence, focusing on the Mediterranean region of Egypt. Floods in coastal areas can be extremely destructive natural hazards resulting in societal and economical damage. In recent years, Earth Observation data, particularly from the Copernicus Programme, has proven invaluable for taking appropriate measures before, during and after flooding events, as it provides a wealth of information on various aspects such as inundated areas, land use, elevation profile and population density. By integrating Artificial Intelligence techniques, we can further enhance the analysis and decision-making processes necessary to prevent or minimise loss in flooding events.

In this tutorial, we will delve into the various methodologies, tools, and applications of Earth Observation data from the Copernicus Programme, with a focus on the Sentinel satellites, for assessing and managing flooding events in Mediterranean Egypt. We will explore how Artificial Intelligence can be harnessed to process and analyze these vast datasets, extracting valuable insights and automating complex tasks. Through hands-on examples and case studies, you will learn how to apply these techniques to real-world scenarios, enabling you to contribute to the ongoing efforts in battling these extreme events.

The topics of floods prediction, monitoring and disaster management are vast. As is the use of Earth Observation for these tasks. This tutorial aims to serve as a starting point and inspiration for further studies.


<a name="learning-objectives"></a>
# Learning Objectives

By the end of this notebook you should be able to:

1. Load and visualise multi-band satellite data — Sentinel-1 SAR, Sentinel-2 multispectral, and a digital elevation model — as NumPy arrays, and explain why a raw satellite image cannot simply be displayed as an RGB picture.
2. Turn large geospatial rasters into patch-based training data for a convolutional network, and explain why the train/validation/test split must happen *before* patching.
3. Train a small convolutional network to predict building density from Sentinel-2 reflectance, and read its training and validation loss curves for signs of over- and underfitting.
4. Evaluate the result both as a regression (RMSE, MAE, MSE) and as a thresholded map (precision, recall, F1, IoU), and explain why accuracy alone is misleading on an unbalanced dataset.
5. Explain what building density is a proxy for, where that proxy breaks down, and who is most likely to be missed by it.
6. Measure the energy and carbon cost of your own training run, and describe how exposure maps feed into flood-risk and disaster-response decisions — and what they must not be used to decide.


<a name="climate-impact"></a>
# Climate Impact
Flooding is one of the most destructive natural hazards on the Earth that creates major economic and societal harm by damaging homes, disrupting critical infrastructure and degrading farmlands according to the [report published by United Nations Office for Disaster Risk Reduction](https://www.undrr.org/publication/human-cost-disasters-overview-last-20-years-2000-2019).
<br><br>
It can be challenging to connect floods to climate change due to the complex interplay of numerous factors, including both natural weather conditions and human actions. Additionally, historical data on floods is limited, making it difficult to compare past occurrences to current trends. However, as the IPCC (Intergovernmental Panel on Climate Change) noted in its [special report on extremes](https://www.ipcc.ch/site/assets/uploads/2018/03/SREX-Chap3_FINAL-1.pdf), there is clear indication that climate change has influenced several water-related variables, such as rainfall and snowmelt, that contribute to floods. While climate change may not directly cause floods, it does exacerbate many of the underlying factors. Global warming will keep increasing also river flood risk in the future resulting in more damage every year according to the [JRC](https://joint-research-centre.ec.europa.eu/system/files/2020-09/05_pesetaiv_river_floods_sc_august2020_en.pdf).
<br><br>

**Alexandria and climate risks** <br>
> Alexandria, located on the Mediterranean coast, is the second-largest city in Egypt and has a population of approximately 5.5 million. Alexandria and the Nile Delta are among the most vulnerable areas in the world to climate change. The UN’s Intergovernmental Panel on Climate Change predicts that global sea levels could rise by as much as 68cm by 2050, flooding parts of Alexandria and causing saltwater intrusion into the groundwater. It would also cause buildings to collapse and salination of farmland in the nearby Nile Delta region destroying livelihoods and triggering internal displacement. Reports suggest that even 50cm of sea-level rise would threaten 2 million people
in Alexandria, including Al Max. Increased temperatures driven by climate change are already affecting biodiversity in the Mediterranean Sea.

Continue reading about Alexandria and climate risks in the [Climate and mobility case study January 2023: Alexandria, Egypt: Al Max](https://reliefweb.int/report/egypt/climate-and-mobility-case-study-january-2023-alexandria-egypt-al-max).
<br>

For these reasons adequate flood risk assessment tools are essential to mitigate damage and save lives. Comprehensive flood risk assessments do not only take the chance of flooding into account but also the possible damages to land and the vulnerability of the people living there. It involves an analysis of which interventions would be most successful in reducing overall harm.
<br>

Satellite imagery and the methods to process them are invaluable tools in the ongoing effort to predict flood risk, proceed with damage control and evacuation in case of a flooding event and motivate policy and contruction works to prevent damage in the future.
<br>

In this tutorial we focus on how we can map surface water and the impact on human lives. However, the tools used to process satellite imagery and prepare them for machine learning tasks are very general and can be used for many other climate science problems.

<a name="target-audience"></a>
# Target Audience

*   Climate scientist seeking to get an introduction to machine learning tools. More specifically how to apply convolutional neural networks to satellite data.
*   Data scientist with little background in remote sensing but interested to explore how satellite data can be processed and used as training data for convolutional neural networks.

The audience of this notebook do not need to be experts neither in AI nor in climate science. There will however be ample oppurtunity for more experienced readers to dive deeper into the data and develop more complex AI pipelines.



<a name="background-and-prereqs"></a>
# Background & Prerequisites


## Population mapping and sea water flood risk assessment in coastal regions

**Population and vulnerable area mapping** is the process of creating a map of the number of people living in specific geographic areas and taking into account the land use as well. This includes for example identifying if the area is a farmland or heavily urbanised and mapping of schools, hospitals and other critical infrastructure. This information is essential for a variety of purposes, including emergency management planning, infrastructure development, and environmental conservation. By mapping the population and vulnerable areas in coastal regions, planners can identify areas where high population density may exacerbate flood risks and take measures to reduce the likelihood of damage to high priority areas such as hospitals and densely populated regions.

**Flood susceptibility maps** are maps describing the flooding tendencies of certain geographic areas based on its physical characteristics. This can include topographical, geographical, and meteorological factors (such as altitude, slope, lithology, land use, and rainfall). Based on these characteristics, areas can be classified as high risk or low risk.

**Flood inundation maps** on the other hand represent the extent of a flood after or during an event has occurred. These maps show what areas were effectively flooded. These maps help with disaster management, damage assessment and evacuation planning.

Access to this data is essential for making critical decisions before, during and after flooding events.
It provides a strong basis to motivate the building of dams and surge protection barriers.

## The role of satellite data in flood risk assessment.

Remote sensing data can be a great help to create these population maps, inundation maps and flood susceptibility maps.

Satellites play a crucial role in remote sensing, as they can capture images of large areas from space. **High-resolution multispectral images** obtained from satellites can be used to estimate population density and land use. This information is useful for disaster response planning and identifying vulnerable areas.

Satellites equipped with **Synthetic Aperture Radar** (SAR) instruments can create waterbody maps before and after flooding events. SAR is very sensitive to water, which makes it an effective tool for monitoring water levels and detecting changes in waterbodies. These maps can help in understanding the extent of flooding, the severity of flood damage, and in identifying areas at risk of flooding.

LiDAR satellites or a constellation of SAR satellites can provide information to create **Digital Elevation Models** (DEMs). A DEM is a 3D model of the topographic surface of an area, excluding trees, buildings, and any other objects on the surface. DEMs extremely useful for flood modelling and watershed analysis.







<a name="software-requirements"></a>
# Software Requirements

On Colab, make sure to **enable a GPU runtime** (should be enabled automatically, otherwise go to 'Runtime' in the upper taskbar and select 'change runtime type'. With a GPU the models will train much faster and their will be more RAM available for processing.

The Python version at the time of this submission in Colab = Python 3.10.11.

**The tools.**

* PyTorch
* Numpy
* Matplotlib
* Buteo
* CodeCarbon

Google Colab comes preinstalled with PyTorch and NumPy and matplotlib, so we only need to install buteo and codecarbon.

**Buteo** is a toolbox designed to simplify the process of working with geospatial data for Deep Learning. It includes tools for reading, writing, and processing geospatial data, as well as tools for creating labels from vector data and generating patches from geospatial data. Buteo makes it easy to ingest data, create training data, and perform inference on geospatial data. <br>
[Documentation](https://casperfibaek.github.io/buteo/buteo.html)

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.7936577.svg)](https://doi.org/10.5281/zenodo.7936577)

**CodeCarbon** is a lightweight library that estimates the electricity a piece of code consumes and converts it into CO₂-equivalent emissions. It samples the power draw of the CPU, GPU, and RAM while your code runs, turns that into kilowatt-hours, and multiplies by the carbon intensity of the electricity grid your machine is plugged into. We use it to put a number on what this notebook itself costs — a habit worth keeping in any climate-related ML project. Part II does the same, so you can compare the two.<br>
[Documentation](https://docs.codecarbon.io/)

In [ ]:
# Install buteo
!pip install buteo==0.9.15 --upgrade -q

In [ ]:
# Install CodeCarbon, to measure the energy and emissions of this notebook.
!pip install codecarbon==3.2.9 -q

In [ ]:
import buteo as beo
import numpy as np
import torch
import os

In [ ]:
#@title This cell sets matplotlib parameters and defines plotting functions
# Data visualization
# This cell sets matplotlib parameters and defines plotting functions.
# You can safely ignore this cell as it only impacts printing functions.

# These functions are just for printing
# in colab.
import matplotlib
import matplotlib.patheffects as path_effects
from matplotlib import pyplot as plt
from collections.abc import Iterable

# Reset Matplotlib parameters to their default values
matplotlib.rcdefaults()

# Lets set some default pyplot parameters to make our plots look pretty.
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (16, 12)
plt.rcParams['figure.subplot.wspace'] = 0.05
plt.rcParams['image.interpolation'] = 'bilinear'

# Match the plot background to the Colab theme. Outside Colab (e.g. running
# this notebook locally) there is no such theme, so we fall back to light mode.
try:
    from google.colab import output
    is_dark = output.eval_js('document.documentElement.matches("[theme=dark]")')
except ImportError:
    is_dark = False

if is_dark:
    COLORMODE = "#38383838" # Are you using colab in darkmode?
else:
    COLORMODE = "#FFFFFFFF" # Or whitemode?

# Custom function for creating subplots with a specified face color
def custom_subplots(*args, facecolor=COLORMODE, size=None, **kwargs):
    fig, axes = plt.subplots(*args, **kwargs)

    if size is not None:
        fig.set_size_inches(size[0], size[1])

    # Make sure axes is a list of axes objects
    if not isinstance(axes, Iterable) or isinstance(axes, np.ndarray) and axes.ndim == 0:
        axes = [axes]

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1)

    # Reduce the distance between the plotted images in the width
    plt.subplots_adjust(wspace=0.05)

    # Change the background color of the whole plot to grey
    fig.set_facecolor(facecolor)  # RGBA tuple for grey, 100% transparent if supported

    axes = axes[0] if len(axes) == 1 else axes

    return fig, axes

<a name="download-data"></a>
# Download the data

Everything in this notebook runs on two data products, both hosted publicly and free to use:

| What | Files | Where it ends up | Used for |
|---|---|---|---|
| **78 training tiles** | `patches_middle-east.zip`, which unpacks to `label_*.tif`, `s2_*.tif`, `s1_*.tif`, `dem_*.tif` | `/content/CCAI_FLOODS_TUTORIAL_DATA/` | Training and testing the building-density model |
| **Full Alexandria scene** | `S2.tif` (required), `S1.tif` (Part II) | `/content/` | Predicting over the whole city at the end of the notebook |

There are several gigabytes in total, so the download takes a few minutes. We provide two routes — run **one** of them, then the verification cell at the end of this section:

* **Option 1 — Google Drive.** Try this first; it is the fastest and fetches only the files you need.
* **Option 2 — Zenodo.** A single archive with everything, used as a backup when the Drive links hit their download quota.

**A note on where the files live**: everything is written under `/content/`, which is the local disk of the Colab virtual machine. It is wiped whenever the runtime is recycled or you disconnect for a while — if that happens, re-run this section. Nothing is written to your own Google Drive and you are never asked to mount it.


## Option 1 — Google Drive

In [ ]:
# All the paths used by this notebook are defined here, in one place.
# Everything lives on the Colab VM's local disk (/content), not in your own Drive.

# The 78 training tiles: label_*.tif, s1_*.tif, s2_*.tif, dem_*.tif
data_folder = "/content/CCAI_FLOODS_TUTORIAL_DATA"

# The full Alexandria scene, used for inference at the end of the notebook.
PATH_S2 = "/content/S2.tif"  # Required by this notebook.
PATH_S1 = "/content/S1.tif"  # Not used in Part I; downloaded here ready for Part II.

# Our own outputs (the trained model and the predicted rasters) go here.
FOLDER_PRED = "/content/CCAI_FLOODS_TUTORIAL_DATA_PREDICTIONS/"

os.makedirs(data_folder, exist_ok=True)
os.makedirs(FOLDER_PRED, exist_ok=True)

In [ ]:
# Download the archive with the 78 training tiles.
# gdown keeps the file name stored in Drive, so this lands at
# /content/patches_middle-east.zip
!gdown https://drive.google.com/uc?id=1N-vOm_rBPXM4CbXTdHhKfjYSeJd93sL0

In [ ]:
# Unpack the tiles into data_folder (/content/CCAI_FLOODS_TUTORIAL_DATA/).
# The .tif files sit at the root of the archive, so after this the tiles are
# directly at /content/CCAI_FLOODS_TUTORIAL_DATA/label_0.tif, s2_0.tif, ...
!unzip -oq /content/patches_middle-east.zip -d /content/CCAI_FLOODS_TUTORIAL_DATA/

Now the full Alexandria scene. Again `gdown` preserves the file names, so these two downloads land at `/content/S1.tif` and `/content/S2.tif`.

`S2.tif` is the one Part I needs — it is what we run the trained model over at the end of the notebook. `S1.tif` is not used here; it is downloaded so it is already in place for the exercises and for Part II. Skip that cell if you are short on disk space or time.


In [ ]:
# S1.tif -> /content/S1.tif (optional in Part I, needed in Part II)
!gdown https://drive.google.com/uc?id=1LrakuW_RVA8KVsls4NyesnG4pFmxu7eS

In [ ]:
# S2.tif -> /content/S2.tif (required: full-scene prediction at the end)
!gdown https://drive.google.com/uc?id=1lrvPBJXHYmA6Xcd9bklNRHjcP9e_zQvQ

## Option 2 — Zenodo (backup)

The Google Drive links are quota-limited: when many people run the tutorial at the same time, Drive can refuse to serve the files. The same data is archived on Zenodo, which has no such limit:

**Alexandria — People and Water dataset**, DOI [10.5281/zenodo.7937444](https://doi.org/10.5281/zenodo.7937444)

It is a single 5.0 GB archive, `Alexandria_S12DEM.zip`, containing `S1.tif`, `S2.tif`, `DEM_Orientation.tif` and the 78 training tiles. Uncomment and run the two cells below: the first downloads the archive, the second sorts its contents into exactly the same paths that Option 1 produces, so the rest of the notebook works unchanged. Expect this to be slower than the Drive route — it downloads everything, including data Part I does not use.


In [ ]:
# !wget -O /content/Alexandria_S12DEM.zip https://zenodo.org/record/7937444/files/Alexandria_S12DEM.zip

In [ ]:
# import glob
# import shutil
# import zipfile
#
# # Extract to a staging folder first. The internal layout of the archive may
# # differ from the Drive downloads, so we sort the files into place ourselves
# # rather than assuming a folder structure.
# STAGING = "/content/zenodo_extract"
# with zipfile.ZipFile("/content/Alexandria_S12DEM.zip", "r") as zip_ref:
#     zip_ref.extractall(STAGING)
#
# # The 78 tiles are named label_*.tif / s1_*.tif / s2_*.tif / dem_*.tif.
# # They belong in data_folder.
# for path in glob.glob(os.path.join(STAGING, "**", "*.tif"), recursive=True):
#     name = os.path.basename(path)
#     if name.split("_")[0] in ("label", "s1", "s2", "dem"):
#         shutil.move(path, os.path.join(data_folder, name))
#
# # The full scenes keep their own names at the top level of /content/.
# for path in glob.glob(os.path.join(STAGING, "**", "*.tif"), recursive=True):
#     name = os.path.basename(path)
#     if name in ("S1.tif", "S2.tif", "DEM_Orientation.tif"):
#         shutil.move(path, os.path.join("/content", name))
#
# shutil.rmtree(STAGING, ignore_errors=True)

## Check that the data is where the notebook expects it

Whichever option you used, run the cell below before continuing. It counts the tiles and confirms that the full scene is in place.


In [ ]:
# Verify the download.
from glob import glob

print(f"Tiles in {data_folder}:")
for kind in ["label", "s2", "s1", "dem"]:
    n_files = len(glob(os.path.join(data_folder, f"{kind}_*.tif")))
    status = "OK" if n_files == 78 else "<-- expected 78, re-run the download above"
    print(f"  {kind + '_*.tif':<12} {n_files:>3} files   {status}")

print("\nFull scene:")
for path, required in [(PATH_S2, True), (PATH_S1, False)]:
    if os.path.exists(path):
        print(f"  {path}   {os.path.getsize(path) / 1e9:.2f} GB   OK")
    else:
        note = "needed for the full-scene prediction" if required else "only needed for Part II"
        print(f"  {path}   MISSING   <-- {note}")


<a name="data-description"></a>
# Data Description

Copernicus is the Earth Observation programme headed by the European Commission in partnership with ESA. It provides accurate, timely and easily accessible information to improve the management of the environment, understand and mitigate the effects of climate change and ensure civil security. We will be using the freely available data from the Sentinel missions and the Copernicus DEM.




## Project data

The project data comprises of 78 locations covering Alexandria and the surrounding Nile delta in Egypt, each encompassing several square kilometers.

For each location, the following data are available:

**Sentinel-1**: There are 2 available bands which are the VV and VH bands. VV is the mode that transmits vertical waves and receives vertical waves to create the SAR image while VH is the mode that transmits vertical waves and receives horizontal waves. The data has already been despeckled and processed to contain values in $dB$.

**Sentinel-2**: There are 9 available bands. These bands are: 2-Blue, 3-Green, 4-Red, 5-RedEdge1, 6-RedEdge2, 7-RedEdge3, 8-NIR, 11-SWIR1, 12-SWIR2. Notice that band 8A-RedEdge is not supplied. These images are not normalised yet, but we will see how to work and visualise them later on.

**CopDEM**: The Copernicus DEM has 4 channels. The actual elevation in meters is stored in the 4th channel. The first 2 channels store the direction (aspect in sin/cos) the slope is facing while the 3rd channel has the actual slope. All the four channels are in the range [0-1] with channel 4 normalised to the height of Mt. Everest.

**Buildings labels**: This is the ground truth used for the building density prediction task (building density serves as proxy for population density). The labels are expressed of number of squared meters of building on a given pixel. Values are between 0-100 $m^2$ and for a resolution of 10$m$ this reflect the percentage of coverage. This data is a combination of the Google Open Buildings dataset, OSM buildings and manual labeling.
<br><br>
Further information about the imagery sources are below:

## Sentinel-1: Synthetic Aperture Radar (SAR) data
SENTINEL-1 is an imaging radar mission providing continuous all-weather, day-and-night imagery at C-band at a resolution of about 10m every 12 days. Sentinel-1 is a phase-preserving dual polarisation SAR system. It can transmit a signal in either horizontal (H) or vertical (V) polarisation, and then receive in both H and V polarisations.

SAR sensors are able to detect flooding because flat surfaces reflect (acts as a specular reflector) the signal away from the sensor, decreasing the amount of returned radiation. This causes relatively dark pixels in radar data for water areas which contrast with non-water areas.

Speckle is a general phenomenon in SAR imagery caused by the interaction of the out-of-phase waves reflected from a target resulting in a salt-and-pepper pattern. Presence of speckle in SAR images degrades the interpretability of the land features in the data. Speckle removal is necessary for quantitative, analysis and there are various filters but there exists a tradeoff between speckle removal and resolution. The images we use have already been despeckled and processed to dB values.

## Sentinel-2: Multispectral data

SENTINEL-2 is a wide-swath, high-resolution, multi-spectral imaging mission, supporting Copernicus Land Monitoring studies, including the monitoring of vegetation, soil and water cover, as well as observation of inland waterways and coastal areas. Most importantly for us is the MultiSpectral Instrument (MSI), which samples 13 spectral bands: four bands at 10 metres, six bands at 20 metres and three bands at 60 metres spatial resolution. These include the Blue, Green and Red bands which correspond to how humans see the world but also Near Infrared (NIR) and Short-wave Infrared (SWIR) which can be used to monitor vegetation, geological features and much more.

Sentinel-2 images are very valuable for population density estimations and the mapping of vulnerable areas. The infrared bands are effective to differentiate between different types of land use while the Red band is used for mapping man-made structures. The high resolution RGB images can be used to detect schools, hospitals and other vulnerable areas.

The bands in this tutorial are ordered like this:

0. Blue
1. Green
2. Red
3. RedEdge 1
4. RedEdge 2
5. RedEgde 3
6. Near-Infrared
7. SWIR 1
8. SWIR 2

<img src='https://drive.google.com/uc?id=1Uj6bTFDFMpn_s_ksgn4WAQKYK-Ct9jMD' width="600"/><br>
<em>RGB image from the MSI instrument. East Jerusalem.</em>
<br><br>

## Copernicus DEM: Digital Elevation Model data

The Copernicus DEM is a Digital Surface Model (DSM) that represents the surface of the Earth including buildings, infrastructure and vegetation. The Copernicus DEM provides digital elevation maps for Egypt at a resolution of 30m. Data to create the DEM were acquired through the TanDEM-X mission.

As one can image, DEM’s are often used for flood prediction. DEMs can be used to create flood inundation maps, which show how water will spread across the landscape during a flood event. By combining a DEM with hydrological modeling, it is possible to simulate flood events and create maps that show which areas will be affected by flooding and to what extent. DEMs can also be used to identify natural features that can affect flooding, such as ridges, valleys, and drainage basins.

<br>
<img src='https://drive.google.com/uc?id=1XaZh1Z_Q6nw-3RW9IpHAUfEprtSEKiGR' width="600"/><br>
<em>Aspect-slope image derived from DEM. East Jerusalem. </em>

## Where the files are on disk

| Path | Contents | Used for |
|---|---|---|
| `/content/CCAI_FLOODS_TUTORIAL_DATA/label_*.tif` | 78 building-density labels, 1 band, 0–100 m² per pixel | Training targets |
| `/content/CCAI_FLOODS_TUTORIAL_DATA/s2_*.tif` | 78 Sentinel-2 tiles, 9 bands | Model input |
| `/content/CCAI_FLOODS_TUTORIAL_DATA/s1_*.tif` | 78 Sentinel-1 tiles, 2 bands (VV, VH) | Exercises (multi-modal input) |
| `/content/CCAI_FLOODS_TUTORIAL_DATA/dem_*.tif` | 78 Copernicus DEM tiles, 4 bands | Exercises |
| `/content/CCAI_FLOODS_TUTORIAL_DATA/train.npz`, `val.npz`, `test.npz` | *Written by this notebook*: normalised patches, split by tile | Training, validation and evaluation |
| `/content/S2.tif` | The full Sentinel-2 scene over Alexandria | Prediction over the whole city |
| `/content/S1.tif` | The full Sentinel-1 scene | Part II |
| `/content/CCAI_FLOODS_TUTORIAL_DATA_PREDICTIONS/` | *Written by this notebook*: `model_01.pt`, `prediction_*.tif` | Outputs to download and open in QGIS |

The tiles and the full scene are not disjoint: some of the 78 tiles fall inside the area covered by `S2.tif`. Keep that in mind when you look at the final full-scene map — part of it covers pixels the model was trained on, so it is a demonstration of large-scale inference, not an independent evaluation.

## Dataset datasheet

A summary of the data you are about to train on. Read it before trusting any map you produce from it.

| | Sentinel-1 tiles | Sentinel-2 tiles | Copernicus DEM tiles | Building labels |
|---|---|---|---|---|
| **What it is** | C-band SAR backscatter in dB, VV and VH (2 bands) | Multispectral reflectance, 9 bands (Blue … SWIR2), uint16 digital numbers | Aspect (sin, cos), slope, and elevation (4 bands, all scaled to 0–1) | Building area per pixel, 0–100 m² |
| **Source** | Copernicus Sentinel-1 (ESA) | Copernicus Sentinel-2 (ESA) | Copernicus DEM (ESA) | Google Open Buildings + OpenStreetMap + manual labelling |
| **Spatial resolution** | 10 m | 10 m (the 20 m bands resampled to 10 m) | 30 m, resampled | 10 m |
| **Coverage** | 78 tiles over Alexandria and the surrounding Nile delta, a few km² each | the same 78 tiles | the same 78 tiles | the same 78 tiles |
| **Temporal coverage** | One acquisition per tile | One acquisition per tile, not necessarily the same date as the S1 tile | Static | Compiled from sources of varying dates |
| **Preprocessing** | Despeckled, calibrated to dB; nodata filled with 0 | None (raw DN); we clip to 0–10 000 and scale in this notebook | Already normalised | Clipped to 0–100, NaNs set to 0 |
| **License** | Free and open (Copernicus data policy) | Free and open (Copernicus data policy) | Free and open | Google Open Buildings: CC BY 4.0 / ODbL; OpenStreetMap: ODbL |
| **Distribution** | Zenodo DOI [10.5281/zenodo.7937444](https://doi.org/10.5281/zenodo.7937444) | same | same | same |

**Known issues to keep in mind:**

* **Building density is a proxy for population, not a measurement.** A warehouse, an office block, and a crowded apartment building look much the same from above. Any statement about *people* derived from these labels inherits that gap.
* **Label noise, unevenly distributed.** The footprints come from automatically-derived and crowd-sourced sources. They are most often missing or misaligned in informal settlements and rapidly growing peri-urban areas — which are frequently the places with the highest flood exposure. The model will be least reliable exactly where the stakes are highest.
* **One region, few dates.** Everything is Alexandria and the Nile delta. A model trained here will not transfer freely to other cities, climates, or building styles without retraining and re-validation.
* **The test split is geographic but not independent.** We hold out whole tiles (better than holding out patches), but neighbouring tiles share land cover and acquisition conditions, so the reported metrics are optimistic relative to a genuinely new region.
* **Nodata is filled with zeros**, which are indistinguishable from genuine zeros (no buildings, or zero backscatter).
* **S1/S2 temporal mismatch**: the radar and optical tiles are not always from the same date, which matters if you extend the model to use both.

<a name="methodology"></a>
# Methodology

We are going to investigate water and population using Sentinel 1, 2, and the CopDEM. The methodology will be kept simple, but there is a lot of room for data exploration and the full dataset for the mediterranean region of Egypt is available. The methods will be divided into 3 steps. The last 2 steps will be done in a seperate notebook because of memory constraints within Colab.

1. Estimation of population and structure density using a Convolutional Neural Network (Part I).
2. Mapping water using an index based approach for Sentinel 2 (Part II).
3. Using the water index (NDWI) from the previous step, we will train a Convolutional Neural Network to estimate water using Sentinel 1. This is highly useful in the case of flooding, which is usually accompanied by poor weather conditions, which make the use of spectral instruments impossible (Part II).

## Constants

Let's gather the settings we need throughout the notebook in one place. Setting `SEED` makes the run reproducible; the rest control how the data is cut into patches and how the model is trained. Every one of them is worth experimenting with — and the carbon tracker in the next cell will tell you what your experiments cost.


In [ ]:
# ML training constants. Please explore adjusting these - on Colab this cell
# renders as an interactive form, so you can change the values without editing code.
EPOCHS = 10 #@param {type:"slider", min:1, max:25, step:1}
PATCH_SIZE = 32 #@param {type:"raw"}
BATCH_SIZE = 16 #@param {type:"raw"}
N_OFFSETS = 3 #@param {type:"raw"}
LEARNING_RATE = 0.001 #@param {type:"number"}
SEED = 17 #@param {type:"raw"}

# PATCH_SIZE: training patches of 32x32 pixels - small enough to keep RAM use modest.
# BATCH_SIZE: patches per batch. N_OFFSETS: extra patch grids offset by (8,8), (16,16)
# and (24,24), so each pixel is sampled several times in different positions.

# Seed the random number generators so that runs are reproducible.
# (Full determinism on a GPU would additionally require
# torch.use_deterministic_algorithms(True), at some cost in speed.)
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## Measuring the carbon cost of our own compute

This notebook is about the climate — so it is only fair to ask what the notebook itself costs. Training and running models consumes electricity, and electricity carries emissions. For a small model like ours the number will be tiny, but the *habit* is what matters: reporting energy and emissions alongside accuracy is becoming standard practice in machine-learning papers, and several venues now ask for it explicitly.

We use [CodeCarbon](https://docs.codecarbon.io/) for this. It works in three steps:

1. While your code runs, it samples the power draw of the CPU, GPU, and RAM (in watts).
2. Power × time gives the energy consumed, in kilowatt-hours (kWh).
3. Energy × the **carbon intensity** of your local electricity grid (grams of CO₂-equivalent per kWh) gives the emissions. That second factor is not a detail: the same kWh emits several times more CO₂ in a coal-heavy grid than in one dominated by hydro, wind, or nuclear power.

The cell below starts a tracker and defines two small helpers, `track_start` and `track_stop`, which we wrap around the two expensive phases of this notebook: **training the model** and **running inference over the whole Alexandria scene**. At the end of the notebook we print a summary. If you would rather not track anything, set `TRACK_EMISSIONS` to `False` — the helpers then quietly do nothing and the rest of the notebook is unaffected.


In [ ]:
#@title Start the carbon tracker { display-mode: "form" }
TRACK_EMISSIONS = True #@param {type:"boolean"}

from codecarbon import EmissionsTracker

# If you re-run this cell, stop the previous tracker first so we do not leave a
# measurement thread running in the background.
if globals().get("emissions_tracker") is not None:
    try:
        emissions_tracker.stop()
    except Exception:
        pass

emissions_tracker = None
emission_tasks = {}  # phase name -> measurement, filled in by track_stop()

if TRACK_EMISSIONS:
    try:
        emissions_tracker = EmissionsTracker(
            project_name="ccai-flood-part1-people",
            output_dir=FOLDER_PRED,      # emissions.csv lands next to the predictions
            output_file="emissions.csv",
            measure_power_secs=5,        # sample the hardware every 5 seconds
            log_level="error",           # keep the notebook output readable
            allow_multiple_runs=True,    # cells get re-run in notebooks
        )
        emissions_tracker.start()
    except Exception as error:
        print(f"CodeCarbon could not start ({error}). Continuing without tracking.")
        emissions_tracker = None


def track_start(phase_name):
    """Start measuring a named phase. Does nothing if tracking is switched off."""
    if emissions_tracker is not None:
        emissions_tracker.start_task(phase_name)


def track_stop(phase_name):
    """Stop measuring the current phase, remember it, and print its footprint."""
    if emissions_tracker is None:
        return None

    measurement = emissions_tracker.stop_task()
    emission_tasks[phase_name] = measurement

    print(
        f"{phase_name}: {measurement.duration:.1f} s, "
        f"{measurement.energy_consumed * 1000:.2f} Wh, "
        f"{measurement.emissions * 1000:.2f} g CO2eq"
    )
    return measurement


# Data Exploration


There are 78 tiles of data, each containing the Digital Elevation Model (In Orientation Format), Sentinel-1, Sentinel-2, and building labels.

**Lets investigate the data:**

To turn the data given tif-format into a NumPy array that is easy to manipulate, we use the `raster_to_array` function from the buteo library.

In [ ]:
print("Shapes are in the form (height, width, channels)")

# data_folder was defined in the download section above:
# /content/CCAI_FLOODS_TUTORIAL_DATA

example_label_path = os.path.join(data_folder, "label_3.tif")
example_label = beo.raster_to_array(example_label_path)

example_dem_path = os.path.join(data_folder, "dem_3.tif")
example_dem = beo.raster_to_array(example_dem_path)

example_s1_path = os.path.join(data_folder, "s1_3.tif")
example_s1 = beo.raster_to_array(example_s1_path)

example_s2_path = os.path.join(data_folder, "s2_3.tif")
example_s2 = beo.raster_to_array(example_s2_path)

example_s2_RGB_path = os.path.join(data_folder, "s2_3.tif")
example_s2_RGB = beo.raster_to_array(example_s2_path, bands=[3, 2, 1]) # Select only RGB in that order.


In [ ]:
print(f"{example_label.shape}: Label shape.")
print(f"{example_dem.shape}: Digital Elevation Model shape.")
print(f"{example_s1.shape}: Sentinel 1 shape.")
print(f"{example_s2.shape}: Sentinel 2 shape.")
print(f"{example_s2_RGB.shape}: Sentinel 2 RGB shape.")

In [ ]:
# To read the metadata of the tiles use the raster_to_metadata function
example_label_meta = beo.raster_to_metadata(example_label_path)
for idx, (key, value) in enumerate(example_label_meta.items()):

    # Some of these strings are long, Lets show only the first parts.
    val = str(value) if len(str(value)) < 50 else str(value)[:50] + "..."
    print(f"{key}: {val}")

## Visualising Labels

Visualise the labels which contain the building density ranging from 0-100$m^2$ per pixel.


In [ ]:
# Lets see what the building labels looks like!
# remember that the value of the labels reflects the percentage of building coverage in the given pixel.
# That means the values of the labels go from [0, 100] inclusive. The brighter the pixel, the more of it
# is covered by structures.
fig, ax1 = custom_subplots()
ax1.imshow(example_label[:, :, 0], vmin=0, vmax=100, cmap="magma")
plt.show()
fig.clf()

## Visualising hyperspectral images

Visualising hyperspectral data is not always an easy task. We are used to see images displayed in RGB (Red-Green-Blue) and 8-bit, which typically have values in the range 0-255. To get to this point from the original hyperspectral image (which comes as 16 bit images) we need to select the rights bands and process them before we can display a nice RGB image. Here is one way to do so!

**Note**: *For some optical satellites, RGB bands might not be available*

In [ ]:
# Rendering hyperspectral images as RGB can be troublesome. Here is a methodology
# Lets define it as a function so we can reuse it later.

# Clip the top and bottom 2% of the data to not normalise to outliers.
def clip_to_quantiles(arr, q_min=0.02, q_max=0.98):
    return np.clip(arr,
        np.nanquantile(arr, q_min),
        np.nanquantile(arr, q_max),
    )

def render_s2_as_rgb(arr):
    # If there are nodata values, lets cast them to zero.
    if np.ma.isMaskedArray(arr):
        arr = np.ma.getdata(arr.filled(0))

    # Select only Blue, green, and red.
    # We copy as float32, so that we neither modify the caller's array in place
    # nor lose precision doing integer division on the original uint16 data.
    rgb_slice = arr[:, :, 0:3].astype(np.float32, copy=True)

    # Clip the data to the quantiles, so the RGB render is not stretched to outliers,
    # Which produces dark images.
    for c in [0, 1, 2]:
        rgb_slice[:, :, c] = clip_to_quantiles(rgb_slice[:, :, c])

    # We want an uint8 RGB render, so we normalise each layer by dividing with
    # the maximum value in the image. Then we multiply it by 255 (the max of
    # uint8) to be in the normal RGB range.
    for c in [0, 1, 2]:
        rgb_slice[:, :, c] = (rgb_slice[:, :, c] / np.amax(rgb_slice[:, :, c])) * 255.0

    # We then round to the nearest integer and cast it to uint8.
    rgb_slice = np.rint(rgb_slice).astype(np.uint8)

    return rgb_slice


In [ ]:
# Lets take it for a spin.
fig, ax1 = custom_subplots()
ax1.imshow(render_s2_as_rgb(example_s2_RGB))
plt.show()
fig.clf()

## Visualising SAR data

The sentinel 1 data has two polarisations, Vertical Vertical and Vertical Horizontal. A common way of visualising SAR imagery as RGB is to do the following: `Blue: VV, Green: VH, Red: VV/VH` but for now, lets just view
one of the polarisations (VV).

In [ ]:
# The sentinel 1 data has two polarisations, Vertical Vertical and Vertical Horisontal.
# Here we only visualise the VV component.
fig, ax1 = custom_subplots()
ax1.imshow(example_s1[:, :, 0], vmin=-20, vmax=5, cmap="magma")
plt.show()
fig.clf()

## Visualising DEM data

The DEM data contains 4 channels that contain information on the elevation profile.

The first 3 channels contain the orientation of the slope and the actual slope itself. Plotting these channels results in an aspect-slope image.
 The 4th channel contains the actual elevation. When plotted this gives us the elevation profile.

In [ ]:
# Aspect-slope image

# The terrain data is already normalised to what is sometimes called the orientation.
# This is the aspect plotted as a 2D circle, and the slope.
fig, ax1 = custom_subplots()

# We normalise the DEM for visualisation to highlight the contrast in the image.
for c in [0, 1, 2]:
    example_dem[:, :, c] = example_dem[:, :, c] / example_dem[:, :, c].max()

ax1.imshow(example_dem[:, :, 0:3])
plt.show()
fig.clf()

In [ ]:
# Elevation profile

# The elevation values themselves are in the fourth band, normalised to the
# height of mt. everest.
fig, ax1 = custom_subplots()
ax1.imshow(example_dem[:, :, 3] / example_dem[:, :, 3].max(), cmap="viridis")
plt.show()

# Predicting building and proxy population density

Our first task is to analyse the regions and find out which places have the highest density of vulnerable locations and where most of the population is concentrated.

For this task we will train a convolutional neural network on the hyperspectral data from Sentinel-2. For the task of building density prediction we have labels which we can use to train the model and evaluate the results.

## Preparing the data for our convolutional neural network
When working with computer vision tasks, it is often necessary to process large amounts of image data. This can be a challenging task, particularly when working with high-resolution images, which can contain millions of pixels. To make this task more manageable, images are typically divided into smaller sub-regions or patches.

These patches can be generated in a variety of ways, such as by using sliding windows or by randomly selecting regions of the image. We are going to use the buteo library for this. The patches are then typically grouped into batches, which are fed into the deep learning model during training. The major benefit of having patches is to reduce the amount of memory required to process the image data and to make it easier to work with.

It is important to be aware that during the process of creating patches some regions in the original image will be sampled multiple times, if overlaps are used. This means patches are not always independent of each other and might partially overlap. This is especially important to take into account when creating the test set, as it is essential that it is not contaminated with training data.

We can easily create the patches by using the `array_to_patches` function from the buteo library while specifying the size of the resulting arrays.


In [ ]:
# Generate patches for a single image. We can choose the size of the resulting patches. Automatically can create offsets.
patches_label = beo.array_to_patches(example_label, 64)
patches_rgb = beo.array_to_patches(example_s2, 64)

# Patch, height, width, channel.
print(patches_label.shape, patches_rgb.shape)

In [ ]:
# We now have 1972 images with height 64 and width 64.
fig, (ax1, ax2) = custom_subplots(nrows=1, ncols=2, size=(6, 6))

# interpolation is set explicit to nearest, smart smoothing at pixel level does not make sense.
ax1.imshow(patches_label[400, :, :, 0], cmap="magma", interpolation="nearest")
ax2.imshow(render_s2_as_rgb(patches_rgb[400, :, :, :]), interpolation="nearest")
plt.show()

## Create train, validation and test sets

A machine learning model needs to be trained on a set of data to learn from it. But to know whether it has learned something that *generalises* — rather than memorised the training patches — we need data it was never trained on. We use three splits:

* **Training set** — what the model learns from.
* **Validation set** — never trained on, but evaluated after every epoch. It tells us *during* training whether the model is still improving or has started to overfit.
* **Test set** — touched once, at the very end, to report final performance.

We set aside 10% of the tiles for validation and another 10% for testing.

**Where the split happens matters.** We split *before* cutting the data into patches, at the level of whole tiles. If we split afterwards, at the patch level, the overlapping patch grids would place the same pixels in more than one split, and our "held-out" scores would partly be measuring data the model had already seen. This is one of the most common and most expensive mistakes in geospatial machine learning: it produces excellent numbers and a model that disappoints in the field.

Even with a tile-level split our held-out data is not fully independent — every tile comes from the same region, sensor, and season, so a held-out tile still resembles the training tiles far more than a genuinely new city would. See the [model card](#model-card) for what that means for the numbers you are about to get.


In [ ]:
# Now lets create all the data we need to train our model!
from glob import glob
from tqdm import tqdm

# Lets randomly select 10% of our tiles for validation and another 10% for testing.
# It is important that we do it at this level instead of the patch level.
# Since we are using "overlaps", doing the selection at the patch level would
# result in contamination as the same pixel can appear in more than one split.

# Set the seeds, to make it replicatable. SEED, PATCH_SIZE and N_OFFSETS were all
# defined in the constants cell near the top of the notebook.
np.random.seed(SEED)

# We draw both hold-out sets in one go, so that they cannot overlap each other.
# replace=False matters: with replacement we would draw the same tile twice and
# silently end up with fewer held-out tiles than we asked for.
N_HOLDOUT = 78 // 10
holdout_indices = np.random.choice(78, 2 * N_HOLDOUT, replace=False)
val_indices = holdout_indices[:N_HOLDOUT]
test_indices = holdout_indices[N_HOLDOUT:]

print(f"Validation tiles ({len(val_indices)}): {sorted(val_indices)}")
print(f"Test tiles       ({len(test_indices)}): {sorted(test_indices)}")

# Lists to hold our patches
training_label = []
training_s1 = []
training_s2 = []
training_dem = []

validation_label = []
validation_s1 = []
validation_s2 = []
validation_dem = []

testing_label = []
testing_s1 = []
testing_s2 = []
testing_dem = []

paths_labels = sorted(glob(os.path.join(data_folder, "label_*.tif")))
paths_s1 = sorted(glob(os.path.join(data_folder, "s1_*.tif")))
paths_s2 = sorted(glob(os.path.join(data_folder, "s2_*.tif")))
paths_dem = sorted(glob(os.path.join(data_folder, "dem_*.tif")))

# Read and order the tiles in the temporary folder.
for image in tqdm(zip(
    paths_labels,
    paths_s1,
    paths_s2,
    paths_dem
), total=len(paths_labels), ncols=120):
    path_label, path_s1, path_s2, path_dem = image

    # Get the name and number of the patches
    label_name = os.path.splitext(os.path.basename(path_label))[0]
    img_idx = int(label_name.split("_")[1])

    # Get the data from the tiles
    arr_label = beo.raster_to_array(path_label, filled=True, fill_value=0.0)

    # Handle any potential errors
    np.clip(arr_label, 0.0, 100.0, out=arr_label)
    arr_label[np.isnan(arr_label)] = 0.0

    # Read the tiles
    arr_s1 = beo.raster_to_array(path_s1, filled=True, fill_value=0.0)
    arr_s2 = beo.raster_to_array(path_s2, filled=True, fill_value=0.0)
    arr_dem = beo.raster_to_array(path_dem, filled=True, fill_value=0.0)

    # Generated the patches
    patches_label = beo.array_to_patches(arr_label, PATCH_SIZE, n_offsets=N_OFFSETS)
    patches_s1 = beo.array_to_patches(arr_s1, PATCH_SIZE, n_offsets=N_OFFSETS)
    patches_s2 = beo.array_to_patches(arr_s2, PATCH_SIZE, n_offsets=N_OFFSETS)
    patches_dem = beo.array_to_patches(arr_dem, PATCH_SIZE, n_offsets=N_OFFSETS)

    # Sanity check to ensure that the right images were chosen.
    assert patches_label.shape[0:3] == patches_s1.shape[0:3] == patches_s2.shape[0:3] == patches_dem.shape[0:3], "Patches do not align."

    if img_idx in test_indices:
        testing_label.append(patches_label)
        testing_s1.append(patches_s1)
        testing_s2.append(patches_s2)
        testing_dem.append(patches_dem)
    elif img_idx in val_indices:
        validation_label.append(patches_label)
        validation_s1.append(patches_s1)
        validation_s2.append(patches_s2)
        validation_dem.append(patches_dem)
    else:
        training_label.append(patches_label)
        training_s1.append(patches_s1)
        training_s2.append(patches_s2)
        training_dem.append(patches_dem)

# Merge the patches back together
training_label = np.concatenate(training_label, axis=0)
training_s1 = np.concatenate(training_s1, axis=0)
training_s2 = np.concatenate(training_s2, axis=0)
training_dem = np.concatenate(training_dem, axis=0)

validation_label = np.concatenate(validation_label, axis=0)
validation_s1 = np.concatenate(validation_s1, axis=0)
validation_s2 = np.concatenate(validation_s2, axis=0)
validation_dem = np.concatenate(validation_dem, axis=0)

testing_label = np.concatenate(testing_label, axis=0)
testing_s1 = np.concatenate(testing_s1, axis=0)
testing_s2 = np.concatenate(testing_s2, axis=0)
testing_dem = np.concatenate(testing_dem, axis=0)

print("Training patches:   ", training_label.shape, training_s1.shape, training_s2.shape, training_dem.shape)
print("Validation patches: ", validation_label.shape, validation_s1.shape, validation_s2.shape, validation_dem.shape)
print("Test patches:       ", testing_label.shape, testing_s1.shape, testing_s2.shape, testing_dem.shape)

## Normalisation strategies

Normalization is a common preprocessing step in computer vision tasks, and it refers to the process of rescaling the pixel values of an image to have a common scale. The main reason for normalizing data in computer vision tasks is to reduce the impact of the scale of the input features on the learning process of the model.


**Sentinel-1**: The current data is expressed in dB. The data is first clipped to the range (-5 dB, 35dB). Afterwards each value is normalised to (0,1) by subtracting the mean and dividing by the stardard deviation.

**Sentinel-2**: The Sentinel-2 images are uint16 with values ranging from 0 to 65535 which are called digital numbers (DN). The physical value derived from this is the reflectance defined as DN/10.000. The reflectance is usually between 0-1 but can be higher due to surface or cloud effects.  By convention Sentinel 2 data is clipped to the range (0,10.000). Afterwards it is normalised to (0,1) as well.

**CopDEM**: The channels in the DEM are already normalised. The elevation (4th channel) is normalised with respect to the heighest point on earth which is  mt. Everest. The slope (3rd band) is also normalised with respect to a slope of 90°. The first two bands contain the orientation of the slope. Because the orientation is a cyclical variable (0° = 360°), we need two channels to encode and  normalise this information. This can be done by storing the sine of the orientation in the first band and the cosine in the second band. To keep the range between (0,1) the final result is:

$$ \text{channel 1} = [sin(\text{orientation})+1]/2 $$
$$ \text{channel 2} = [cos(\text{orientation})+1]/2 $$


In [ ]:
# The next step is normalising the data.
# By convention Sentinel 2 data is normalised by dividing with 10000.0
# Sentinel 1 is more tricky. The current data is dB, which ranges from ~-35 to 5,
# The DEM is already normalised.

# Scalers also return a dictionary of the stats, so the same scaler can be reused.
# Note that all three splits are scaled with the *same* fixed ranges. The ranges
# come from the physics of the sensors, not from the data, so there is nothing to
# leak from the held-out splits back into training.
s1_train, _statdict = beo.scaler_truncate(training_s1, -35.0, 5.0)
s1_val, _statdict = beo.scaler_truncate(validation_s1, -35.0, 5.0)
s1_test, _statdict = beo.scaler_truncate(testing_s1, -35.0, 5.0)

s2_train, _statdict = beo.scaler_truncate(training_s2, 0.0, 10000.0)
s2_val, _statdict = beo.scaler_truncate(validation_s2, 0.0, 10000.0)
s2_test, _statdict = beo.scaler_truncate(testing_s2, 0.0, 10000.0)

# If the labels are not already float32, cast them to float32.
labels_train = training_label.astype(np.float32, copy=False)
labels_val = validation_label.astype(np.float32, copy=False)
labels_test = testing_label.astype(np.float32, copy=False)

Let's save all our work to disk as NumPy arrays. That way, at any moment in time, we could reload the data again and use it to train the model. A cool thing to know about saving NumPy arrays is that later on, when loading the data again, the data can be kept on disk, and only necessary bits can be loaded into memory upon access. This can be very useful when dealing with large datasets.

In [ ]:
# Lets save the data to our temporary folder and do some house cleaning.
# This takes about 2 minutes, so go make some coffee!
np.savez_compressed(os.path.join(data_folder, "train.npz"), x_s1=s1_train, x_s2=s2_train, x_dem=training_dem, y=labels_train)
np.savez_compressed(os.path.join(data_folder, "val.npz"), x_s1=s1_val, x_s2=s2_val, x_dem=validation_dem, y=labels_val)
np.savez_compressed(os.path.join(data_folder, "test.npz"), x_s1=s1_test, x_s2=s2_test, x_dem=testing_dem, y=labels_test)

# Memory cleaning
del s1_train, s2_train, training_dem, labels_train
del s1_val, s2_val, validation_dem, labels_val
del s1_test, s2_test, testing_dem, labels_test

## Model definition

We start by building a very simple convolutional network that can already obtain reasonable results. The model will take as input an image containing the 9 spectral bands from Sentinel-2 (normalised and split up into patches of 32x32). The output will be the building density of the pixel ranging between 0-100. This means we are doing regression task as we are trying to predict specific values.

The architecture is simple. First there is an encoder block where the number of channels is increased to 128. Because of the padding, the resolution of the output channels is unchanged. In the decoder block, there are extra convolutions that bring the number of channels back to a single one that contains the building density.

To help the model we force the values to lie between in the range 0-100 since any other values will never be correct.




In [ ]:
import torch.nn as nn

# This is a Simple Convolutional Neural Network.
class SimpleConvNet(nn.Module):
    def __init__(self, in_channels, output_min, output_max):
        super(SimpleConvNet, self).__init__()
        self.output_min = output_min
        self.output_max = output_max

        # An encoder without a bottleneck
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(128)
        )

        # Simple decoder
        self.decoder = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 1, kernel_size=3, padding=1)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)

        # Lets help the network
        x = torch.clamp(x, self.output_min, self.output_max)
        return x

This is a very simple model. It does not have a bottleneck, skip connections, attention *layers* or other modern features. Please investigate ways of improving this network. A good place to start would be a bottleneckand and skip connections - Maybe a residual block? Check out InceptionResNet and ConvNext for modern convolutional architectures.

A key benefit of the current model is that it takes up **less than 1mb of space**.

In [ ]:
# Define the model
input_channels = 9 # Sentinel 2 initially.

# Since we know the labels will always be [0.0, 100.0], we can help the network.
model = SimpleConvNet(input_channels, 0.0, 100.0)

# EPOCHS, BATCH_SIZE and LEARNING_RATE were set in the constants cell near the top
# of the notebook - go back there to change them.
print(f"Training for {EPOCHS} epochs, batch size {BATCH_SIZE}, learning rate {LEARNING_RATE}.")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


It is important to be aware that there are different conventions regarding the data format. More specifically TensorFlow uses the convention of channels last while PyTorch uses channels first. When using libraries with different conventions one must take care to change the format when necessary.

In [ ]:
# Create the dataloader
from torch.utils.data import Dataset, DataLoader

class NumpyDataset(Dataset):
    def __init__(self, x_train, y_train, data_is_channel_last=False):
        if data_is_channel_last:
            x_train = beo.channel_last_to_first(x_train)
            y_train = beo.channel_last_to_first(y_train)

        self.x_train = torch.from_numpy(x_train).float()
        self.y_train = torch.from_numpy(y_train).float()

    def __len__(self):
        return len(self.x_train)

    def __getitem__(self, index):
        x = self.x_train[index]
        y = self.y_train[index]
        return x, y

In [ ]:
# Load the data. We open the archive once and pull both arrays out of it.
with np.load(os.path.join(data_folder, "train.npz")) as train_npz:
    x_train = train_npz["x_s2"] # Note: Initially we only load the S2 Data.
    y_train = train_npz["y"]

# Ready the data for pytorch
def callback(x, y):
    return (
        torch.from_numpy(x).float(),
        torch.from_numpy(y).float(),
    )

# The validation split: never trained on, evaluated after every epoch.
with np.load(os.path.join(data_folder, "val.npz")) as val_npz:
    x_val = val_npz["x_s2"]
    y_val = val_npz["y"]

# Create the dataset and DataLoader
dataset = NumpyDataset(x_train, y_train, data_is_channel_last=True)
dataset_val = NumpyDataset(x_val, y_val, data_is_channel_last=True)
# num_workers is the number of subprocesses loading data in parallel. A Colab VM
# only has a couple of CPU cores, so asking for many more workers than that costs
# memory and start-up time without loading anything faster.
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=2)

# No shuffling for validation - we are only measuring, not learning.
dataloader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=2)

print(f"{len(dataset):>7,} training patches")
print(f"{len(dataset_val):>7,} validation patches")

## Metrics

In machine learning, a metric is a measure used to evaluate the performance of a model on a given task. Metrics are used to quantify how well a model is performing and to compare the performance of different models. <br><br>

**Regression** <br>
The choice of metric depends on the task at hand. In our case we are trying to predict the density of buildings at every given pixel. A very common metric for regression tasks is the Mean Squared Error (MSE). For a given image the MSE is calculated as follows:

$$MSE = \frac{1}{n} \sum_{i=1}^n (y_{pred,i} - y_{label,i})^2 $$

Where $n$ is the number of pixels in the image, $y_{pred,i}$ is the i-th pixel from the prediction and  $y_{label,i}$ is the i-th pixel from the label. <br>
In a similar fashion there is the Mean Average Error (MAE) and Root Mean Squared Error (RMSE) which are expressed in the same units as the label:
$$MAE = \frac{1}{n} \sum_{i=1}^n |y_{pred,i} - y_{label,i}| $$
<br>
$$RMSE = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_{pred,i} - y_{label,i})^2}.$$

<br><br>

**Classification** <br>
One could also cast the problem to a binary classification task. This changes the problem from predicting the building density (ranging from 0-100) to predicting if buildings are present or not (0 or 1). In this case we should recast the labels based on a threshold. For example if the building density is greater than 1, we say buildings are present and the new label is 1, else the new label is 0.

For a binary classification task we can use accuracy as a metric. Accuracy is the percentage of correctly classified examples. For a given image the accuracy would be:

$$ accuracy = \frac{\# \text{correctly classified pixels}}{\# \text{pixels in the image}}$$

<br><br>
*accuracy is a very simple metric for classification problems, can you find more nuanced metrics?*


## Train model

We now train the model while optimising for MSE loss.
Inspect how the training loss behaves during training.

In [ ]:
# Lets train!
import torch.optim as optim

# Our loss function
criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Move model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

# We record the mean loss per epoch so we can plot learning curves afterwards.
train_losses = []
val_losses = []

track_start("Model training")

# Training loop
for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    running_loss = 0.0

    # Initialize the progress bar for training
    train_pbar = tqdm(dataloader, total=len(dataloader), ncols=120)

    for i, (inputs, targets) in enumerate(train_pbar):
        # Move inputs and targets to the device (GPU)
        inputs, targets = inputs.to(device), targets.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Compute loss
        loss = criterion(outputs, targets)

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

        # Print statistics
        current_loss = loss.item()

        running_loss += current_loss
        mean_loss = running_loss / (i + 1)

        # Update the training bar
        train_pbar.set_description(f"Epoch: {epoch+1:03d}/{EPOCHS:03d}")
        print_dict = { "loss": f"{mean_loss:4f}" }

        train_pbar.set_postfix(print_dict)

    train_losses.append(running_loss / len(dataloader))

    # --- Validation (no gradients, no weight updates) ---
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for val_inputs, val_targets in dataloader_val:
            val_inputs, val_targets = val_inputs.to(device), val_targets.to(device)
            running_val_loss += criterion(model(val_inputs), val_targets).item()

    val_losses.append(running_val_loss / len(dataloader_val))
    print(f"Epoch {epoch+1:03d}: train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}")

track_stop("Model training")

# Switch the model to evaluation mode now that training is done. This matters:
# our model contains BatchNorm layers, which behave differently in the two modes.
# While training they normalise using the statistics of the current batch; in
# eval mode they use the running averages accumulated over training. If we forget
# this, every prediction below would depend on which other patches happen to sit
# in the same batch - and the test metrics would be quietly wrong.
model.eval()


## Loss curves

A single number at the end of training tells you little — plotting the training and validation loss per epoch is the quickest way to see *how* the model learned.

* If both curves keep falling together, the model is still learning generalisable structure: train longer.
* If the training loss keeps falling while the validation loss flattens or turns upward, the model has started to **overfit** — memorising training patches instead of learning transferable features. The epoch where the validation curve turns is where you should have stopped.
* If both curves flatten early at a high value, the model is **underfitting**: it lacks the capacity (or the training time) to capture the pattern at all.

This is the diagnostic that a test set alone cannot give you, because the test set is only looked at once, at the end.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
epochs_range = range(1, len(train_losses) + 1)
ax.plot(epochs_range, train_losses, marker="o", label="Training loss")
ax.plot(epochs_range, val_losses, marker="o", label="Validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.set_title("Training and validation loss")
ax.legend()
plt.show()


In [ ]:
# Save the model in case you want to reuse it.
torch.save(model.state_dict(), os.path.join(FOLDER_PRED, "model_01.pt"))

del dataset, dataloader, dataset_val, dataloader_val

<a name="results-and-discussion"></a>
# Results & discussion


## Metrics on test set

Metrics are a very important tool to explore model performance. They allow to get a quick summary of the global performance and compare different model architectures, training schemes, data augmentation, etc.

Let's take a look how the trained model performs on the test set for the metrics we defined earlier. We also try to recast the labels to a binary classification problem and see how well it scores in this setup.

In [ ]:
# Lets predict our tests dataset and inspect the results.
# Clear GPU memory (only meaningful if we actually have a GPU).
if device.type == "cuda":
    torch.cuda.empty_cache()

# The model was put into eval mode at the end of training - see the note there.
model.eval()

with torch.no_grad():
    x_test = beo.channel_last_to_first(np.load(os.path.join(data_folder, "test.npz"))["x_s2"])
    y_test = beo.channel_last_to_first(np.load(os.path.join(data_folder, "test.npz"))["y"])
    y_pred = model(torch.from_numpy(x_test).float().to(device)).cpu().detach().numpy()

print("Building density metrics:")
print("\tRMSE:", np.sqrt(np.mean((y_test - y_pred) ** 2)))
print("\tMAE:", np.mean(np.abs(y_test - y_pred)))
print("\tMSE:", np.mean((y_test - y_pred) ** 2))

# Recast problem to classification
threshold = 1 # If 1m^2 of the pixel is covered by buildings, we say that buildings are present
y_test_classification = np.where(y_test > threshold, 1, 0)
y_pred_classification = np.where(y_pred > threshold, 1, 0)

print("\n\nClassification metrics:")
print("\tAccuracy:", np.mean(y_test_classification == y_pred_classification))

**Be critical about metrics**

Metrics are a very important tool to explore model performance, but they can sometimes be misleading as well.

Notice that the MSE on the test set is much higher than the loss values you saw at the end of training. Some gap is expected — this is data the model has never seen. But compare it against the validation curve you just plotted. If validation loss tracked training loss closely and the test MSE is *still* much higher, then simple overfitting is not the whole story: the test tiles differ from the training tiles in some systematic way, or a handful of very bad predictions are dominating the average. MSE squares its errors, so it is extremely sensitive to outliers. Let's plot the distribution of the squared error to see which it is.

Sometimes models find clever ways to optimise the training metric without actually learning the desired properties. An example of this can be in unbalanced datasets. In our dataset for example there will always be much more pixels not containing any building than the other way around. If the model would optimise accuracy it could get very good accuracy by lazily predicting that no pixels contain buildings!

Let's dig a bit deeper into these metrics and see how they compare to simple baseline predictions.


In [ ]:
# Distribution of the SE loss
SE = (y_test - y_pred) ** 2
plt.figure(figsize=(8,4))
_,_,_ = plt.hist(SE.flatten(), bins=[100*i for i in range(20)])
plt.ylabel('No of pixels', size='x-large')
plt.xlabel('Squared Error', size= 'x-large')

# Some very simple baselines to compare the model too
baseline_pred_0 = np.zeros(y_pred.shape)
baseline_pred_mean = np.ones(y_pred.shape)*np.mean(y_train)

# To score the baselines as classifiers we must threshold them exactly the way we
# thresholded the model's predictions above - otherwise we are comparing a class
# label against a raw density value.
baseline_0_classification = np.where(baseline_pred_0 > threshold, 1, 0)
baseline_mean_classification = np.where(baseline_pred_mean > threshold, 1, 0)

print("Metric for baseline model that always predicts 0")
print("\tRMSE:", np.sqrt(np.mean((y_test - baseline_pred_0) ** 2)))
print("\tMAE:", np.mean(np.abs(y_test - baseline_pred_0)))
print("\tMSE:", np.mean((y_test - baseline_pred_0) ** 2))
print("\taccuracy:", np.mean(y_test_classification == baseline_0_classification))

print(f"\n\nMetric for baseline model that always predicts the mean of the train labels ({np.mean(y_train):.2f}m^2).")
print("\tRMSE:", np.sqrt(np.mean((y_test - baseline_pred_mean) ** 2)))
print("\tMAE:", np.mean(np.abs(y_test - baseline_pred_mean)))
print("\tMSE:", np.mean((y_test - baseline_pred_mean) ** 2))
print("\taccuracy:", np.mean(y_test_classification == baseline_mean_classification))

# Look at that first accuracy score. A model that predicts "no buildings, anywhere"
# scores extremely well, because most pixels genuinely contain no buildings. That is
# the unbalanced-dataset trap the text above warns about, in numbers.

## Beyond accuracy: precision, recall, F1 and IoU

You just saw the problem in numbers: a model that predicts "no buildings anywhere" scores a very high accuracy, because most pixels genuinely contain no buildings. So let's use metrics that cannot be fooled that way. We threshold both the predictions and the labels into binary built-up / not-built-up masks and count four outcomes:

* **Precision** — of the pixels we flagged as built-up, how many really are? Low precision = false alarms.
* **Recall** — of the actual built-up pixels, how many did we find? Low recall = missed buildings.
* **F1** — the harmonic mean of precision and recall, a single number that punishes lopsided models.
* **IoU (Intersection over Union)** — the overlap between the predicted and true built-up areas; the standard metric for segmentation tasks.

None of these are improved by predicting "nothing everywhere": that model has zero true positives, so its precision, recall, F1 and IoU all collapse to zero, no matter how many empty pixels it gets right.

The trade-off between precision and recall is not just academic. If this map is used to decide where to send flood warnings, false alarms erode trust and waste resources, while missed buildings mean warnings that never reach people in the water's path. Which error is worse depends on how the map is used — and that is a decision for the people running the response, not for the model.


In [ ]:
# Binary built-up detection, using the same threshold as above.
# y_test_classification and y_pred_classification were computed in the previous cell.
tp = int(np.sum((y_pred_classification == 1) & (y_test_classification == 1)))
fp = int(np.sum((y_pred_classification == 1) & (y_test_classification == 0)))
fn = int(np.sum((y_pred_classification == 0) & (y_test_classification == 1)))
tn = int(np.sum((y_pred_classification == 0) & (y_test_classification == 0)))

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

print(f"Built-up detection at a threshold of {threshold} m2 per pixel:")
print(f"  Precision: {precision:.3f}")
print(f"  Recall:    {recall:.3f}")
print(f"  F1:        {f1:.3f}")
print(f"  IoU:       {iou:.3f}")

print()
print(f"  Built-up pixels in the test set: {tp + fn:,} of {tp + fp + fn + tn:,} "
      f"({100 * (tp + fn) / (tp + fp + fn + tn):.1f} %)")
print("  ^ that imbalance is why accuracy alone flattered the model.")

# Try changing `threshold` in the cell above and re-running both cells. Watch
# precision and recall move in opposite directions - choosing the threshold IS
# choosing the map.


**It is therefore important to not blindly rely on metrics but inspect the results visually as well.**

In [ ]:
# You can safely ignore this cell for now

# Callback function to predict array using buteo
# The idea behind this function is to provide a way to go from
# Numpy patches in NHWC -> NCHW -> Tensor -> GPU -> PREDICT -> CPU -> Numpy -> NCHW
# That way we can automatically predict large images without having the write tons
# of code.
# The function sets eval mode and disables gradients itself, so it is always safe
# to call, no matter what state the notebook is in when you reach it.
def predict(arr):
    swap = beo.channel_last_to_first(arr)
    as_torch = torch.from_numpy(swap).float()
    on_device = as_torch.to(device)

    model.eval()
    with torch.no_grad():
        predicted = model(on_device)

    on_cpu = predicted.cpu()
    as_numpy = on_cpu.numpy()
    swap_back = beo.channel_first_to_last(as_numpy)

    return swap_back

## Inspect predictions on test set

In [ ]:
# Read and order the tiles in the temporary folder.
# predict the images in the test set
for image in zip(
    sorted(glob(os.path.join(data_folder, "label_*.tif"))),
    sorted(glob(os.path.join(data_folder, "s1_*.tif"))),
    sorted(glob(os.path.join(data_folder, "s2_*.tif"))),
    sorted(glob(os.path.join(data_folder, "dem_*.tif"))),
):
    path_label, path_s1, path_s2, path_dem = image

    label_name = os.path.splitext(os.path.basename(path_label))[0]
    img_idx = int(label_name.split("_")[1])

    # We are only interested in predicting our test dataset.
    if img_idx not in test_indices:
        continue

    # Load the s2 and label data.
    arr_s2 = beo.raster_to_array(path_s2, filled=True, fill_value=0.0)
    arr_label = beo.raster_to_array(path_label, filled=True, fill_value=0.0)

    # Prepare the RGB render as in the previous example
    arr_s2_rgb = beo.raster_to_array(path_s2, filled=True, fill_value=0.0, bands=[3, 2, 1])
    rgb_render = render_s2_as_rgb(arr_s2_rgb)

    # Normalise the S2 data in the same fashion as previously.
    arr_s2, _statdict = beo.scaler_truncate(arr_s2, 0.0, 10000.0)
    arr_s2 = arr_s2.astype(np.float32, copy=False)

    with torch.no_grad():

        # Some tiles are too small to cut into PATCH_SIZE patches. We skip those,
        # but we report what went wrong rather than swallowing every error - a bare
        # 'except: pass' would also hide out-of-memory errors and shape bugs.
        try:
            # Predict the image and merge using the median of predictions.
            predicted = beo.predict_array(
                arr_s2,
                predict,
                tile_size=PATCH_SIZE,
                n_offsets=3,
                merge_method="median",
            )

            # Plot the prediction, the label, and the RGB.
            fig, (ax1, ax2, ax3) = custom_subplots(nrows=1, ncols=3, size=(12, 8))
            ax1.imshow(predicted, vmin=0.0, vmax=100.0, cmap="magma")
            ax2.imshow(arr_label, vmin=0.0, vmax=100.0, cmap="magma")
            ax3.imshow(rgb_render)

            plt.show()

        except (ValueError, RuntimeError) as e:
            print(f"Skipping tile {img_idx}: {e}")

  # From left to right:
  # prediction, building label, RGB Sentinel 2 Render

## Inspect predictions on all of Alexandria

We can inspect how the model performs when predicting building density on the entire region of Alexandria. This includes regions for which we have no labels (so they were not included in either train or test set).

To avoid a RAM overload, we split the scene in 9 chunks (using `raster_to_array_chunks`) and predict them seperately.

In [ ]:
# Lets predict all of alexandria
# Keeps running out of RAM here -> Reduce the size of alexandria dataset or
# Do the calculation in offsets.
# You could make a mosaic of the final images to create the whole scene again.
path_s2 = PATH_S2  # /content/S2.tif, downloaded at the top of the notebook

track_start("Whole-scene inference")

id = 0
predicted_files = []
for arr, offset in beo.raster_to_array_chunks(path_s2, 9, filled=True, fill_value=0.0, cast=np.float32):

    # Normalise the S2 data in the same fashion as previously.
    arr, _statdict = beo.scaler_truncate(arr, 0.0, 10000.0)

    # When running inference on computer vision models, especially small models like this,
    # it is very common to see noise around the edges of the patches. To alleviate this
    # we can predict multiple overlapping patches and weigh edge pixels less. We can also
    # take a robust score of the predictions like 'MAD', 'Median', or 'Olympian', to create
    # nice smooth predictions, even for very small models.
    with torch.no_grad():
        predicted = beo.predict_array(
            arr,
            predict,
            tile_size=PATCH_SIZE,
            n_offsets=3,
            merge_method="median",
            edge_weighted=True,
        )

    # Render an RGB image for comparison
    rgb = render_s2_as_rgb(arr[:, :, 0:3][:, :, ::-1])

    fig, (ax1, ax2) = custom_subplots(nrows=1, ncols=2, size=(12, 8))
    ax1.imshow(predicted, vmin=0.0, vmax=100.0, cmap="magma")
    ax2.imshow(rgb)
    plt.show()

    # Save the prediction
    out_path = beo.array_to_raster(
        predicted,
        reference=path_s2,
        out_path=os.path.join(FOLDER_PRED, f"prediction_{id}.tif"),
        pixel_offsets=offset,
    )

    id += 1
    predicted_files.append(out_path)

track_stop("Whole-scene inference")

# From left to right:
# prediction, RGB Sentinel 2 Render

<a name="model-card"></a>
## Model card

Every model that produces a map should ship with a short statement of what it is and where it breaks. Here is that statement for the model you just trained.

| | |
|---|---|
| **Model** | `SimpleConvNet` — 4 convolutional layers (~150k parameters), no downsampling and no skip connections, output clamped to [0, 100] |
| **Task** | Per-pixel regression of building area (m² per 10 m pixel) from Sentinel-2 reflectance (9 bands) |
| **Training data** | 32×32 px patches with 4 offset grids, taken from the 64 of the 78 tiles not held out for validation or testing; labels as described in the datasheet above |
| **Training setup** | See the constants cell: epochs, batch size, learning rate, and seed — Adam optimizer, MSE loss, validation loss tracked every epoch |
| **Evaluation** | RMSE / MAE / MSE on held-out tiles, plus precision / recall / F1 / IoU after thresholding at 1 m², compared against constant baselines |
| **Intended use** | Education; qualitative mapping of built-up density over this region |
| **Out of scope** | Reporting population counts; other regions, seasons, or sensors without retraining and re-validation; any automated decision about warnings, evacuation, or resource allocation |
| **Known failure modes** | Building density stands in for population, so dense industrial areas read as dense settlement and vice versa; label gaps in informal settlements propagate straight into the predictions; tile-edge inconsistencies (mitigated by overlapping predictions and median merging); bare rock, sand, and greenhouses can look built-up in reflectance; no calibrated uncertainty — a predicted value of 40 is not "40 m² ± something" |

 The validation and test tiles are held out at the tile level, which avoids the patch-overlap contamination described earlier — but they still come from the **same region, sensor, and season** as the training tiles. A held-out tile in the Nile delta resembles a training tile in the Nile delta far more than a tile over Jakarta, Lagos, or Rotterdam would. The metrics you got are therefore an optimistic estimate of what this model would do over a city it has never seen, and the honest way to report them is "held-out tiles, same region", not "test accuracy".


<a name="carbon-cost"></a>
# The Carbon Cost of This Notebook

We measured the two expensive phases — training and whole-scene inference. Let's stop the tracker and see what they cost. The numbers below are specific to *your* run: the hardware Colab allocated you, the number of epochs you chose, and the electricity grid the data centre sits on.


In [ ]:
# Stop the tracker and summarise what this notebook cost.
CAR_G_CO2_PER_KM = 120.0  # An average new car in Europe emits roughly 120 g CO2 per km.

if globals().get("emissions_tracker") is None:
    print("Carbon tracking is off (or the tracker cell was not run), so there is nothing to report.")
else:
    total_kg = emissions_tracker.stop()
    summary = emissions_tracker.final_emissions_data

    print(f"{'Phase':<26}{'Time (s)':>10}{'Energy (Wh)':>14}{'CO2eq (g)':>12}")
    print("-" * 62)
    for phase_name, measurement in emission_tasks.items():
        print(
            f"{phase_name:<26}{measurement.duration:>10.1f}"
            f"{measurement.energy_consumed * 1000:>14.3f}"
            f"{measurement.emissions * 1000:>12.3f}"
        )
    print("-" * 62)
    print(f"{'Whole notebook':<26}{'':>10}{summary.energy_consumed * 1000:>14.3f}{total_kg * 1000:>12.3f}")

    # The grid you are plugged into matters as much as the code you wrote:
    # the same kWh emits ~8x more CO2 in a coal-heavy grid than in a hydro/nuclear one.
    # kg CO2eq / kWh -> g CO2eq / kWh
    intensity = (total_kg / summary.energy_consumed) * 1000 if summary.energy_consumed > 0 else 0.0

    print()
    print(f"CPU:            {summary.cpu_model} ({int(summary.cpu_count)} cores)")
    print(f"GPU:            {summary.gpu_model or 'none detected'} ({int(summary.gpu_count)})")
    print(f"Grid:           {summary.country_name} ({summary.country_iso_code}) - {intensity:.0f} g CO2eq/kWh")
    print(f"Equivalent to:  {total_kg * 1000 / CAR_G_CO2_PER_KM:.2f} km driven by an average car")
    print(f"Full log:       {os.path.join(FOLDER_PRED, 'emissions.csv')}")


### Reading the number

Two things are usually more surprising than the total itself.

**Inference is not free.** Compare the two rows. Training was a one-off; inference over the scene is what an operational service would repeat for every new acquisition, over whole countries, for years. In deployed systems the lifetime cost of inference routinely dwarfs the cost of training. If you scale this notebook from Alexandria to the Mediterranean, that second row is the one that grows.
The same job run on a grid at 700 g CO₂eq/kWh emits roughly eight times what it would on one at 90 g. Choosing a low-carbon region for a training job is often a bigger lever than any optimisation you can make to the code.

### What this measurement does and does not include

CodeCarbon reports an *estimate*, and it is worth being precise about its boundaries:

* It measures the **energy drawn while your code runs** — CPU, GPU, and RAM. Where hardware power sensors are not readable (many virtual machines, most cloud CPUs), it falls back to modelled values based on the processor's rated draw, so treat the figures as an order of magnitude, not a meter reading.
* Grid carbon intensity is a **country- or region-level average**, not the real-time mix at the moment you ran.
* It does **not** include the embodied carbon of manufacturing the hardware, the satellites and ground segment that produced the data, the storage and transfer of that data, or data-centre cooling overhead beyond a default PUE assumption.

So the number is a floor, not a full life-cycle assessment. That is still far more useful than reporting nothing.


None of this argues against doing the work. A few grams of CO₂ to map who is exposed to flooding — against damage measured in lives and billions — is an easy trade. But the trade should be *stated*, not assumed, and it stops being obvious at scale: a foundation model pretrained on global satellite imagery can emit several tonnes. The question to carry forward is not "is this model too expensive?" but "is this the cheapest model that answers the question, and did I say what it cost?"

### Exercises

* Re-run training with more epochs, or a larger `PATCH_SIZE`, and compare the energy per phase. Does accuracy improve in proportion to the extra energy spent?
* Switch the Colab runtime from GPU to CPU and train again. The GPU draws far more power — but finishes much sooner. Which one used less energy overall?
* Open `emissions.csv` in the predictions folder. It has one row per run, with CPU, GPU, and RAM energy broken out. Plot energy against epochs across a few runs.
* Compare your training footprint with Part II's. The two models have almost the same architecture but different input channels and different amounts of data — where does the difference come from?


<a name="predictions-to-decisions"></a>
# From Predictions to Decisions

So far this has been a machine-learning exercise. But building-density maps like the one you just made are produced operationally, and it is worth being explicit about **how they are actually used, and by whom**.

## What you just built, in risk terms

Disaster risk is usually decomposed into three factors, and no single one of them answers the question "who needs help?":

| Factor | Question it answers | Source in this tutorial |
|---|---|---|
| **Hazard** | Where is the water? | Part II (Sentinel-1 → surface water) |
| **Exposure** | What and who is in the water's way? | **This notebook** (Sentinel-2 → building density) |
| **Vulnerability** | Who is least able to cope? | Not in this tutorial — see the limitations below |

Your map is the **exposure** layer. On its own it says nothing about flooding at all: it is a map of where people have built things. It becomes a risk product only when overlaid on a hazard layer. Overlaying them gives the statements that actually drive decisions: *"this many buildings are inside the flooded area"*, *"these neighbourhoods flood first if the water rises"*, *"these roads to the hospital are cut off"*.

## Decision workflows exposure maps feed

* **Preparedness** (before an event): combined with elevation-based flood susceptibility, exposure maps identify high-risk neighbourhoods, informing land-use planning, drainage investment, and evacuation routes.
* **Early warning and anticipatory action** (as an event develops): forecast rainfall plus susceptibility and exposure determine who gets warned and where aid is pre-positioned *before* the flood peaks.
* **Response planning** (during): rapid water maps intersected with exposure show which areas to evacuate first and where to site shelters and distribution points.
* **Recovery and resource allocation** (after): flooded area × building density supports damage assessment, prioritisation of reconstruction funds, and insurance payouts.
* **Filling the map gap**: in many rapidly growing cities there is simply no up-to-date building inventory. A model like this one produces a first approximation from free satellite imagery, in hours, at no marginal cost.

## Who the stakeholders are

* **National and local government**: civil-protection agencies, hydro-meteorological services, and municipalities — the ones who issue warnings and order evacuations.
* **International mapping services**: the [Copernicus Emergency Management Service](https://emergency.copernicus.eu/) and [UNOSAT](https://unosat.org/) produce exactly this kind of rapid exposure and damage mapping, on request, within hours of a disaster.
* **Humanitarian organisations**: IFRC and Red Cross / Red Crescent societies (including forecast-based financing and anticipatory-action programmes), UN agencies, and NGOs deciding where to send teams and supplies.
* **Financial actors**: the World Bank / GFDRR, development banks, and insurers using exposure estimates for risk financing and payouts.
* **Affected communities**: the people the maps are ultimately about — and who are usually *not* in the loop when the maps are made. Keep that in mind when you read the next section.


In [ ]:
# A first "decision product": how much built-up area did we actually map?
PIXEL_AREA_M2 = 10 * 10  # Sentinel pixels are 10 m x 10 m.

total_built_m2 = 0.0
total_scene_m2 = 0.0
for f in predicted_files:
    pred = beo.raster_to_array(f, filled=True, fill_value=0.0)
    total_built_m2 += float(np.sum(pred))            # each value is m2 of building in that pixel
    total_scene_m2 += pred.size * PIXEL_AREA_M2      # each pixel covers 100 m2 of ground

print(f"Mapped built-up area: {total_built_m2 / 1e6:8.1f} km2")
print(f"Scene area:           {total_scene_m2 / 1e6:8.1f} km2")
print(f"Built-up share:       {100 * total_built_m2 / total_scene_m2:8.1f} %")

# Exercise: this is the denominator of an exposure statistic, not the statistic
# itself. In Part II you produce a water mask for the same scene. Intersect the
# two and compute the built-up area that falls *inside* the water. That single
# number - exposed built-up area - is the kind of figure a civil-protection
# agency actually acts on. Who else, from the stakeholder list above, would want
# it, and in what form: a number, a map, or a per-neighbourhood table?


<a name="limitations-responsible-use"></a>
# Limitations & Responsible Use

The section above shows the promise; this one is about the responsibility. A model-based exposure map is **not a neutral or perfect picture of reality** — it is a chain of modelling choices (sensor, labels, patch size, architecture, training region), each of which shapes where the map says buildings are. When such maps influence where warnings, resources, and support are directed, their limitations become people's problems.

**Building density is not population.** This is the single most important caveat in this notebook, and it is baked into the title: we predict *building density* and call it a proxy for *population density*. The proxy breaks in both directions. A warehouse district, a shopping centre, and an industrial estate register as dense built-up area with almost nobody living there at night. A crowded informal settlement of small, low structures registers as sparse. Occupancy rates, household sizes, and building use all vary enormously and none of them are visible from above. If you need people, use a population product built for it (WorldPop, GHSL) — and read *its* limitations too.

**Exposure is not vulnerability.** Counting buildings in a flood zone tells you what is *exposed*, not who can cope. Two equally flooded neighbourhoods can need very different help depending on construction quality, income, age, disability, and access to transport. Building height matters especially: in a deep flood, someone in a single-storey house cannot escape vertically while someone in a multi-storey block can move upstairs and wait for rescue. A map that shows only exposure will systematically understate the needs of the most vulnerable.

**Data gaps and representation.** Building datasets are least accurate exactly where risk is often highest. Informal settlements and rapidly growing peri-urban areas are chronically under-mapped in both Google Open Buildings and OpenStreetMap, and global models are trained mostly on well-mapped regions. Our labels inherit those gaps, and the model learns them: it will under-predict density in precisely the neighbourhoods that are hardest to reach and most likely to be flooded. **The communities missing from the data are missing from the map — and therefore at risk of being missing from the response.**

**Uncertainty.** The model outputs a number per pixel with no calibrated confidence attached: a prediction of 40 is not "40 m² ± something", and it is not a probability. Honest products communicate uncertainty — confidence maps, ranges, masks of known artifacts — instead of presenting one crisp polygon as truth.

**Spatial resolution.** At 10 m per pixel, a map can support neighbourhood-level decisions ("this district is densely built") but not building-level ones ("this house is occupied"). Narrow streets, small structures, and buildings under tree cover are simply invisible. Match the decision to the resolution — never the other way around.

**Generalisation.** The model was trained on one region, one sensor, and essentially one season. Building materials, roof colours, urban form, and vegetation differ enormously between cities; a model trained on Alexandria has no reason to work in Jakarta. Retrain and re-validate, and report where the metrics do *not* apply.

**High-stakes use.** Errors are not symmetric: over-predicting exposure wastes resources and erodes trust; under-predicting it means people who never get warned. When a map influences who gets helped, its errors decide who doesn't. Practical rules of thumb used by operational services:

* Model outputs are **decision support, not decision making** — a human who understands the local context stays in the loop.
* Validate against ground truth (or at least independent data) before operational use, and report metrics honestly — including where they do not apply.
* Ship documentation with every model and dataset (that is what the [model card](#model-card) and the datasheet above are for).
* Ask who is *not* represented in your data — and what your map will do to them.

**Reflection**: look back at your predicted map of Alexandria. If it fed an aid-allocation decision tomorrow, which neighbourhoods would you least trust it in, who would you consult before acting, and what would you refuse to let the map decide on its own?


## Reflections and exercises

* The model used was very simple and had some key limitations. Can you identify what they were? Can you imagine and/or design a better architecture that would outperform the simple model?
* More data is available to run the model, than  what was used. How would you update the model to include SAR and possibly terrain data into the classification as well? Could multi-modality reduce some of the shortcomings of the S2 approach?
* We report precision, recall, F1 and IoU at a threshold of 1 m². Sweep the threshold instead of fixing it, and plot precision against recall. Which operating point would you hand to someone planning an evacuation, and which to someone estimating reconstruction costs?
* Where did the simple model do well and where did not do so well? Reflect on the results.
* We now track a validation loss every epoch, but we do nothing with it. Automate the decision: implement early stopping (halt when validation loss has not improved for N epochs), keep the best checkpoint rather than the last one, or reduce the learning rate on a plateau. Compare the test metrics before and after.
* Our validation and test tiles come from the same region as the training tiles. Build a genuinely harder evaluation: hold out a spatially contiguous block of tiles rather than a random scatter of them. Do the metrics drop? That drop is the part of your score that was regional similarity rather than skill.
* The training loop uses no data augmentation. Part II uses `beo.AugmentationDataset` with rotation, mirroring, and cutmix — port it here. Augmentation makes the training data harder to fit, so watch what it does to the *gap* between the training and validation curves rather than to the training loss alone.
* Using the `buteo.array_to_raster` function you can export the predictions an investigate them in QGIS. It is very important to visually inspect your results and not rely on metrics alone, which can skew your thinking and optimization approaches.

Optional: Uncomment the code below to download files to inspect in QGIS or other GIS software.

In [ ]:
# from google.colab import files

# files.download(predicted_files[5])

<a name="references"></a>
# References

* The human cost of disasters: an overview of the last 20 years: https://www.undrr.org/publication/human-cost-disasters-overview-last-20-years-2000-2019
* Bentivoglio, R., Isufi, E., Jonkman, S. N., and Taormina, R.: Deep learning methods for flood mapping: a review of existing applications and future research directions, Hydrol. Earth Syst. Sci., 26, 4345–4378, https://doi.org/10.5194/hess-26-4345-2022, 2022
* Sentinel-1 data products, https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-1/data-products
* Sentinel-2 data products, https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-2/data-products
* COP-DEM, https://spacedata.copernicus.eu/collections/copernicus-digital-elevation-model
* Alexandria — People and Water dataset (the data used in this tutorial), https://doi.org/10.5281/zenodo.7937444
* Google Open Buildings, https://sites.research.google/open-buildings/
* Mitchell, M. et al. (2019). *Model Cards for Model Reporting.* Proceedings of FAT* 2019. https://arxiv.org/abs/1810.03993
* Gebru, T. et al. (2021). *Datasheets for Datasets.* Communications of the ACM, 64(12). https://arxiv.org/abs/1803.09010
* Courty, B. et al. (2024). *CodeCarbon: Estimate and Track Carbon Emissions from Machine Learning Computing.* https://github.com/mlco2/codecarbon · Documentation: https://docs.codecarbon.io/
* Lacoste, A. et al. (2019). *Quantifying the Carbon Emissions of Machine Learning.* https://arxiv.org/abs/1910.09700 · ML CO2 Impact calculator: https://mlco2.github.io/impact/
* Strubell, E., Ganesh, A., McCallum, A. (2019). *Energy and Policy Considerations for Deep Learning in NLP.* ACL 2019. https://arxiv.org/abs/1906.02243
* Luccioni, A.S., Viguier, S., Ligozat, A.-L. (2023). *Estimating the Carbon Footprint of BLOOM, a 176B Parameter Language Model.* JMLR 24. https://arxiv.org/abs/2211.02001
* Rolnick, D. et al. (2022). *Tackling Climate Change with Machine Learning.* ACM Computing Surveys, 55(2). https://arxiv.org/abs/1906.05433
* Copernicus Emergency Management Service (rapid mapping): https://emergency.copernicus.eu/
* UNOSAT (United Nations Satellite Centre): https://unosat.org/
* WorldPop population data: https://www.worldpop.org/
* GHSL — Global Human Settlement Layer: https://ghsl.jrc.ec.europa.eu/


